# SDG mentions in the Daily Briefings

This notebook visualize the mentions of the 17SDGs in the UN daily briefings.
If an SDG is mentioned is determined by an LLM giving the definition of each 
SDG and saved to a database.

## Set up

Download data to use this notebook
```bash
# Data 1 week of briefings from August 2026 
briefings update 2026-08-01 2026-08-07
briefings init
briefings load 2026-08-01 2026-08-07
```

In [ ]:
from datetime import datetime

import ollama

from un_llm_analysis.database import (
    get_db_connection,
    get_briefing_id_from_date,
    get_briefings_between_dates,
    insert_into_sdg,
    get_sdgs_between_dates,
)

# SDGs

In [ ]:
# https://sdgs.un.org/goals
sdgs = {
    "SDG1": "End poverty in all its forms everywhere",
    "SDG2": "End hunger, achieve food security and improved nutrition and promote sustainable agriculture",
    "SDG3": "Ensure healthy lives and promote well-being for all at all ages",
    "SDG4": "Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all",
    "SDG5": "Achieve gender equality and empower all women and girls",
    "SDG6": "Ensure availability and sustainable management of water and sanitation for all",
    "SDG7": "Ensure access to affordable, reliable, sustainable and modern energy for all",
    "SDG8": "Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all",
    "SDG9": "Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation",
    "SDG10": "Reduce inequality within and among countries",
    "SDG11": "Make cities and human settlements inclusive, safe, resilient and sustainable",
    "SDG12": "Ensure sustainable consumption and production patterns",
    "SDG13": "Take urgent action to combat climate change and its impacts",
    "SDG14": "Conserve and sustainably use the oceans, seas and marine resources for sustainable development",
    "SDG15": "Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss",
    "SDG16": "Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels",
    "SDG17": "Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development",
}

sdg_text = ""
for sdg in sdgs:
    sdg_text += f"* {sdg} {sdgs[sdg]}\n"

sdg_definitions = sdg_text.strip()

In [ ]:
prompt = """**Classification Task:**

Classify the following text into one or more of the following 17 Sustainable Development Goals (SDGs):

**Text to be Classified:**

{text}

**SDG Definitions:**

{sdg_definitions}

**Example Response:**

If the text belongs to one or more of the 17 SDGs, provide a list of relevant goal(s) in the format: ['SDG1', 'SDG5'].

**Edge Case:**

If the text does not belong to any of the 17 SDGs, return 'None'. Do not include any text to your answer, just provide the response in a list.
"""

In [ ]:
start_date = "2026-08-01"
end_date = "2026-08-07"
conn = get_db_connection()
briefings = get_briefings_between_dates(conn=conn, start_date=start_date, end_date=end_date)

In [ ]:
llm = "llama3.2"
conn = get_db_connection()
for briefing in briefings:
    if len(briefing.sections) > 0:
        for i, section in enumerate(briefing.sections, start=1):
            result = ollama.generate(
                model=llm,
                prompt=prompt.format(text=section.text, sdg_definitions=sdg_definitions),
                think=False,
            )

            briefing_id = get_briefing_id_from_date(briefing.briefing_date)

            insert_into_sdg(
                conn=conn,
                briefing_id=briefing_id,
                section_id=briefing_id * 100 + i,
                sdgs=result["response"],
                model=llm,
                created_at=result["created_at"],
            )

conn.close()

# Visualisation

In [87]:
import plotly.express as px

conn = get_db_connection()
llm = "llama3.2"
df = get_sdgs_between_dates(conn=conn, start_date=start_date, end_date=end_date)

df["order"] = df.sdg.apply(lambda x: int(str(x).replace("SDG", "")))
df.sort_values(by="order", inplace=True)

# Prepare and sort data for plot
df_plot = (
    df[df["model"] == llm][["order", "sdg"]]
    .value_counts()
    .reset_index()
    .sort_values("order", ascending=False)
    .set_index("sdg")
    .drop(columns=["order"])
)

x = [x[0] for x in df_plot.values.tolist()]  # frequency of each SDG in the sections.
y = (
    df_plot.index.tolist()
)  # values from the "sdg" column with values: ["SDG1", "SDG2", ..., "SDG17"]

fig = px.bar(
    df,
    x=x,
    y=y,
    labels={
        "x": "# of Mentions in Sections",
        "y": "Sustainable Development Goal",
    },
    orientation="h",
    title=f"17SDGs in {len(df.briefing_id.unique())} Daily Briefings between {str(datetime.strptime(str(df.briefing_id.min()), "%Y%m%d"))[:10]} and {str(datetime.strptime(str(df.briefing_id.max()), "%Y%m%d"))[:10]}",
)
fig.show()